In [0]:
import pandas as pd
import time

def executar_etl_sequencial_avancado(caminho_pasta):
    print("Iniciando o processamento Sequencial Avançado (O Teste de Estresse)...")
    inicio = time.time()

    # 1. Extração
    print("1. Carregando os Gigabytes para a memória (Acompanhe sua RAM!)...")
    df_orders = pd.read_csv(f"{caminho_pasta}/orders_gigante.csv")
    df_items = pd.read_csv(f"{caminho_pasta}/items_gigante.csv")
    df_customers = pd.read_csv(f"{caminho_pasta}/customers_gigante.csv")
    
    # Lendo as tabelas dimensões originais
    df_products = pd.read_csv(f"{caminho_pasta}/olist_products_dataset.csv")
    df_sellers = pd.read_csv(f"{caminho_pasta}/olist_sellers_dataset.csv")

    # 2. Transformação 
    print("2. Tratando Datas e Filtrando...")
    # Filtrar pedidos cancelados 
    df_orders = df_orders[df_orders['order_status'] != 'canceled']
    
    # Converter string para datetime e extrair Ano e Mês 
    df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
    df_orders['ano'] = df_orders['order_purchase_timestamp'].dt.year
    df_orders['mes'] = df_orders['order_purchase_timestamp'].dt.month

    print("3. Executando a cascata de JOINs (Onde a memória chora)...")
    # Juntando as 5 tabelas
    df_merged = pd.merge(df_orders, df_items, on="order_id", how="inner")
    df_merged = pd.merge(df_merged, df_customers, on="customer_id", how="inner")
    df_merged = pd.merge(df_merged, df_products, on="product_id", how="inner")
    df_merged = pd.merge(df_merged, df_sellers, on="seller_id", how="inner")

    print("4. Agregação Complexa (Group By Multidimensional)...")
    # Calcular Faturamento Total, Ticket Médio e Quantidade de Vendas
    # Agrupado por Estado, Ano, Mês e Categoria do Produto
    resultado = df_merged.groupby(
        ['customer_state', 'ano', 'mes', 'product_category_name']
    ).agg(
        faturamento_total=('price', 'sum'),
        ticket_medio=('price', 'mean'),
        quantidade_vendas=('order_id', 'count')
    ).reset_index()

    # Ordenar os resultados do maior faturamento para o menor
    resultado = resultado.sort_values(by=['faturamento_total'], ascending=[False])

    # 3. Carga
    print("5. Guardando o resultado analítico...")
    resultado.to_csv("resultado_analise_avancada_sequencial.csv", index=False)

    fim = time.time()
    tempo_total = fim - inicio

    print("\nProcessamento concluído!")
    print(f"Tempo total de execução (Sequencial): {tempo_total:.2f} segundos")
    print("\nTop 5 Categorias/Estados com maior faturamento:")
    print(resultado.head())

if __name__ == "__main__":
    caminho_dos_dados = "./archive" 
    executar_etl_sequencial_avancado(caminho_dos_dados)

##Codigo para ampliar o dataset 

In [0]:
import pandas as pd
import os

def multiplicar_dataset(caminho_origem, caminho_destino, multiplicador):
    print(f"Lendo {caminho_origem}...")
    df = pd.read_csv(caminho_origem)
    
    print(f"Tamanho original: {len(df)} linhas. Multiplicando por {multiplicador}x...")
    # Multiplica o dataframe concatenando ele mesmo várias vezes
    df_gigante = pd.concat([df] * multiplicador, ignore_index=True)
    
    # Adicionando um sufixo ao ID para simular chaves únicas e forçar o processador
    if 'order_id' in df_gigante.columns:
        df_gigante['order_id'] = df_gigante['order_id'] + "_" + df_gigante.index.astype(str)
        
    print(f"Salvando novo arquivo com {len(df_gigante)} linhas...")
    df_gigante.to_csv(caminho_destino, index=False)
    print(f"Concluído: {caminho_destino}\n")

if __name__ == "__main__":
    pasta = "./archive"
    
    # Vamos multiplicar por 50
    multiplicador = 50 
    
    multiplicar_dataset(f"{pasta}/olist_orders_dataset.csv", f"{pasta}/orders_gigante.csv", multiplicador)
    multiplicar_dataset(f"{pasta}/olist_order_items_dataset.csv", f"{pasta}/items_gigante.csv", multiplicador)
    multiplicar_dataset(f"{pasta}/olist_customers_dataset.csv", f"{pasta}/customers_gigante.csv", multiplicador)